# Netflix Catalog Analysis: A Beginner's Data Analytics Project

**What you will learn:** how to load a dataset with pandas, inspect it, clean it, answer questions with it, and present the results as charts and a short story.

**The dataset:** *Netflix Movies and TV Shows* on Kaggle (file name: `netflix_titles.csv`). Each row is one title, with columns such as `type`, `title`, `director`, `cast`, `country`, `date_added`, `release_year`, `rating`, `duration`, `listed_in` (genres) and `description`.

**Questions we will answer**
1. How many titles are movies vs TV shows?
2. How has the number of releases changed over the years and decades?
3. When were titles added to Netflix, and how old were they when added?
4. Which genres and countries dominate, and how do genres change over time?
5. How long are movies, and how many seasons do TV shows have?
6. Which audience ratings are most common?

**The 5-step analytics workflow we follow**
1. **Ask** - decide what you want to know (the questions above)
2. **Prepare** - load the data and understand what is in it
3. **Clean** - fix errors, handle missing values, reshape messy columns
4. **Analyze** - group, count, compare, chart
5. **Share** - turn findings into a clear story

## How to run this notebook (free, no installation)

1. Download the dataset: go to Kaggle, search for **"Netflix Movies and TV Shows"** (dataset by Shivam Bansal, `shivamb/netflix-shows`), and download `netflix_titles.csv`. A free Kaggle account is needed.
2. Go to [colab.research.google.com](https://colab.research.google.com), choose **File > Upload notebook**, and upload this `.ipynb` file.
3. Run the cells from top to bottom with **Shift + Enter**. When the loading cell asks, upload `netflix_titles.csv`.

**Good to know:** this dataset is a snapshot from around 2021, so it describes *that* catalog, not today's Netflix. Keep that in mind whenever you write conclusions.

## Step 0: Setup

We import the libraries we need:
- **pandas** - tables of data (called DataFrames)
- **matplotlib** and **seaborn** - charts

We also create a small helper function, `save_chart`, that saves each chart as an image in a `charts` folder so you can reuse it in slides or a report.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

NETFLIX_RED = "#E50914"
SERIES_2 = "#2a78d6"

os.makedirs("charts", exist_ok=True)


def save_chart(name):
    """Save the current chart as a PNG (for slides/reports), then show it."""
    plt.tight_layout()
    plt.savefig(f"charts/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 1: Load the data

`pd.read_csv` reads a CSV file into a DataFrame. The first few lines only matter in Google Colab: they open an upload button. On your own computer they are skipped automatically.

In [ ]:
try:
    from google.colab import files
    files.upload()  # choose netflix_titles.csv when the button appears
except ImportError:
    pass  # running locally: the CSV just needs to be in the same folder

df = pd.read_csv("netflix_titles.csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

## Step 2: Understand the data before touching it

Never clean blindly. First answer: *what is in here, and what looks wrong?*

`df.info()` shows each column, its data type, and how many values are not empty.

In [ ]:
df.info()

**How to read this:** if a column has fewer non-null values than the total rows, it has missing data. Notice that `date_added` is stored as `object` (text), not as a real date. We will fix that.

Next, a clearer view of how much is missing in each column.

In [ ]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_%"] = (missing["missing_count"] / len(df) * 100).round(1)
missing.sort_values("missing_count", ascending=False)

Now check for **duplicates** and look at the values inside a few categorical columns. Spotting odd values here is how you discover data problems.

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Repeated show_id values:", df["show_id"].duplicated().sum())
print()
print(df["type"].value_counts())
print()
print(df["rating"].value_counts())

**Things to notice**
- `director`, `cast` and `country` have many missing values.
- Look closely at the `rating` list. In the commonly shared Kaggle version, a few rows contain values like `74 min` or `84 min`. That is a *duration* typed into the *rating* column, a classic data entry error.
- `duration` mixes two units: `"90 min"` for movies and `"2 Seasons"` for TV shows. You cannot do maths on text like that.
- `listed_in` (genres) and `country` hold *several values in one cell*, such as `"Dramas, International Movies"`.

## Step 3: Clean the data

**Golden rule: never edit the original.** We work on a copy called `clean`, so `df` stays untouched and we can always compare before vs after.

### 3.1 Remove stray spaces
Hidden leading/trailing spaces make `" United States"` and `"United States"` look like two different countries.

In [ ]:
clean = df.copy()

text_cols = clean.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    clean[col] = clean[col].str.strip()

### 3.2 Fix values in the wrong column
Where `rating` contains `"min"`, that value really belongs in `duration`. We move it there, then blank out the bad rating.

In [ ]:
bad_rating = clean["rating"].str.contains("min", na=False)
print("Rows where rating actually holds a duration:", bad_rating.sum())

move_to_duration = bad_rating & clean["duration"].isna()
clean.loc[move_to_duration, "duration"] = clean.loc[move_to_duration, "rating"]
clean.loc[bad_rating, "rating"] = pd.NA

clean.loc[bad_rating, ["title", "rating", "duration"]]

### 3.3 Handle missing values
You have three basic choices, and the right one depends on the column:

| Option | When it makes sense |
|---|---|
| **Fill with a label** (e.g. "Unknown") | Text columns where "we don't know" is itself meaningful |
| **Drop the rows** | Only a tiny share of rows are affected and the column is essential |
| **Leave as missing** | Numeric columns where inventing a number would distort results |

Here we label unknown text values and keep everything else as is.

In [ ]:
for col in ["director", "cast", "country"]:
    clean[col] = clean[col].fillna("Unknown")

clean["rating"] = clean["rating"].fillna("Not Rated")

### 3.4 Turn text into real dates
Once `date_added` is a real date, we can pull out the year and month for analysis. `errors="coerce"` turns anything unreadable into a missing value instead of crashing.

In [ ]:
clean["date_added"] = pd.to_datetime(
    clean["date_added"], format="%B %d, %Y", errors="coerce"
)
print("Rows with a missing/unreadable date_added:", clean["date_added"].isna().sum())

clean["year_added"] = clean["date_added"].dt.year
clean["month_added"] = clean["date_added"].dt.month_name()

clean[["title", "date_added", "year_added", "month_added"]].head()

### 3.5 Split `duration` into usable numbers
A regular expression pulls out the number. Then we create two clean numeric columns: minutes for movies, seasons for TV shows.

In [ ]:
parts = clean["duration"].str.extract(r"(\d+)\s*(\w+)")
clean["duration_num"] = pd.to_numeric(parts[0])

clean["movie_minutes"] = clean["duration_num"].where(clean["type"] == "Movie")
clean["tv_seasons"] = clean["duration_num"].where(clean["type"] == "TV Show")

clean[["type", "duration", "movie_minutes", "tv_seasons"]].head(8)

### 3.6 Group ratings into audience categories
There are around a dozen rating codes. Grouping them makes charts easier to read. This is called **feature engineering**: creating a new, more useful column from an existing one.

In [ ]:
rating_groups = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Older Kids", "PG": "Older Kids",
    "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
}
clean["audience"] = clean["rating"].map(rating_groups).fillna("Unrated / Other")

clean["audience"].value_counts()

### 3.7 Add decade and "age when added" columns
- `decade` groups release years (1994 becomes 1990). This is a simple, readable way to see **release-year clusters**.
- `years_before_added` = how many years passed between a title's release and its arrival on Netflix. Negative values are impossible for our purposes, so we blank them.

In [ ]:
clean["decade"] = (clean["release_year"] // 10 * 10).astype(int)

clean["years_before_added"] = clean["year_added"] - clean["release_year"]
clean["years_before_added"] = clean["years_before_added"].where(
    clean["years_before_added"] >= 0
)

clean[["title", "release_year", "decade", "year_added", "years_before_added"]].head()

### 3.8 Split multi-value cells into separate rows
A cell like `"Dramas, Comedies"` holds two genres. To count genres properly we split the text into a list and then **explode** the list so each genre gets its own row.

**Important:** after exploding, one title appears once *per genre*. A title with 3 genres counts 3 times in the `genres` table. That is correct for "which genres are popular" but wrong for "how many titles are there", so always keep both tables in mind.

In [ ]:
genres = (
    clean[["show_id", "type", "release_year", "decade"]]
    .assign(genre=clean["listed_in"].str.split(","))
    .explode("genre")
)
genres["genre"] = genres["genre"].str.strip()
genres = genres.dropna(subset=["genre"])

countries = (
    clean[["show_id", "type"]]
    .assign(country=clean["country"].str.split(","))
    .explode("country")
)
countries["country"] = countries["country"].str.strip()
countries = countries[~countries["country"].isin(["", "Unknown"])]

print("Titles:", len(clean), "| Title-genre pairs:", len(genres), "| Title-country pairs:", len(countries))
genres.head()

### 3.9 Verify the cleaning worked
Always check your work with a before-vs-after comparison. If a number does not move the way you expect, investigate.

In [ ]:
key_cols = ["director", "cast", "country", "rating", "duration", "date_added"]
quality = pd.DataFrame({
    "missing_before": df[key_cols].isna().sum(),
    "missing_after": clean[key_cols].isna().sum(),
})
print(quality)

assert clean["show_id"].is_unique, "show_id should be unique"
assert clean["movie_minutes"].dropna().between(1, 600).all(), "movie minutes look wrong"

clean.to_csv("netflix_clean.csv", index=False)  # save the cleaned data for later
print("\nAll checks passed. Cleaned data saved as netflix_clean.csv")

## Step 4: Analyze and visualize

Each section below follows the same pattern: **question, calculation, chart, what it means**. Notice that chart titles are built from the computed numbers, so they state the actual finding. A good chart title is a conclusion, not just a label.

### Q1. How many movies vs TV shows?

In [ ]:
type_counts = clean["type"].value_counts()
type_share = (type_counts / type_counts.sum() * 100).round(1)
print(type_share)

ax = type_counts.plot(kind="bar", color=[NETFLIX_RED, SERIES_2], rot=0)
ax.bar_label(ax.containers[0], padding=3)
ax.set_title(f"{type_share.idxmax()}s make up {type_share.max():.0f}% of the titles")
ax.set_xlabel("")
ax.set_ylabel("Number of titles")
save_chart("01_movies_vs_tv")

### Q2. How has the number of releases changed by year?
We group by release year and type, count the rows, and use `unstack` to turn types into separate columns (one line per type).

In [ ]:
by_year = (
    clean[clean["release_year"] >= 1990]
    .groupby(["release_year", "type"])
    .size()
    .unstack(fill_value=0)
)
peak_year = by_year.sum(axis=1).idxmax()

ax = by_year.plot(color=[NETFLIX_RED, SERIES_2], linewidth=2.5, figsize=(10, 5))
ax.set_title(f"Titles by release year peaked in {peak_year}")
ax.set_xlabel("Release year")
ax.set_ylabel("Number of titles")
ax.legend(title="")
save_chart("02_releases_by_year")

**Reading tip:** this chart counts titles *that are on Netflix*, not everything ever produced. A catalog reflects licensing and Netflix's own production strategy, so be careful about claiming "the world made more films in year X".

### Q3. Which decades do titles come from? (release-year clusters)

In [ ]:
decade_counts = clean["decade"].value_counts().sort_index()
top_decade = decade_counts.idxmax()

ax = decade_counts.plot(kind="bar", color=NETFLIX_RED, rot=45)
ax.set_title(f"Most titles were released in the {top_decade}s")
ax.set_xlabel("Decade")
ax.set_ylabel("Number of titles")
save_chart("03_titles_by_decade")

### Q4. When were titles added to Netflix, and how old were they?

In [ ]:
added_per_year = clean["year_added"].dropna().astype(int).value_counts().sort_index()
peak_added = added_per_year.idxmax()

ax = added_per_year.plot(kind="line", marker="o", color=NETFLIX_RED, linewidth=2.5)
ax.set_title(f"Most titles were added to Netflix in {peak_added}")
ax.set_xlabel("Year added")
ax.set_ylabel("Number of titles added")
save_chart("04_titles_added_per_year")

gap_by_type = clean.groupby("type")["years_before_added"].median()
print("Median years between release and arrival on Netflix:")
print(gap_by_type)

**Median vs mean:** we used the median because a few very old classics would pull the average up. The median is the "typical middle value" and is more reliable when data has extremes (outliers).

### Q5. Which genres are most common?

In [ ]:
top_genres = genres["genre"].value_counts().head(10).sort_values()

ax = top_genres.plot(kind="barh", color=NETFLIX_RED)
ax.bar_label(ax.containers[0], padding=3)
ax.set_title(f"'{top_genres.index[-1]}' is the most common genre label")
ax.set_xlabel("Number of titles carrying this genre label")
ax.set_ylabel("")
save_chart("05_top_genres")

### Q6. How do the biggest genres change by decade?
A **heatmap** is ideal for comparing two categories at once. Darker cells mean more titles. We keep the top 8 genres and decades from 1980 onward so the picture stays readable.

In [ ]:
top8 = genres["genre"].value_counts().head(8).index

heat = (
    genres[genres["genre"].isin(top8) & (genres["decade"] >= 1980)]
    .groupby(["genre", "decade"])
    .size()
    .unstack(fill_value=0)
    .reindex(top8, fill_value=0)
)

plt.figure(figsize=(9, 5))
sns.heatmap(heat, annot=True, fmt="d", cmap="Reds", cbar_kws={"label": "Titles"})
plt.title("Genre labels by release decade (top 8 genres)")
plt.xlabel("Release decade")
plt.ylabel("")
save_chart("06_genre_decade_heatmap")

### Q7. How long are movies?
A **histogram** shows how values are spread out. The dashed line marks the median.

In [ ]:
movies = clean[clean["type"] == "Movie"]
median_runtime = movies["movie_minutes"].median()

ax = movies["movie_minutes"].plot(kind="hist", bins=30, color=NETFLIX_RED, edgecolor="white")
ax.axvline(median_runtime, color="black", linestyle="--", linewidth=2)
ax.set_title(f"The typical movie runs about {median_runtime:.0f} minutes")
ax.set_xlabel("Runtime (minutes)")
ax.set_ylabel("Number of movies")
save_chart("07_movie_runtime_hist")

print(movies["movie_minutes"].describe().round(1))

**Outliers:** `describe()` shows the minimum and maximum. Extreme values are worth a look. Are they real (a very long film) or data errors?

In [ ]:
print("Longest movies:")
print(movies.nlargest(5, "movie_minutes")[["title", "release_year", "movie_minutes"]])
print()
print("Shortest movies:")
print(movies.nsmallest(5, "movie_minutes")[["title", "release_year", "movie_minutes"]])

**Did movies get longer or shorter over time?** We compute the median runtime per decade. We only use decades with at least 10 movies, because a median based on 2 movies is not trustworthy.

In [ ]:
runtime_stats = movies.groupby("decade")["movie_minutes"].agg(["median", "count"])
runtime_by_decade = runtime_stats[runtime_stats["count"] >= 10]["median"]

ax = runtime_by_decade.plot(kind="line", marker="o", color=NETFLIX_RED, linewidth=2.5)
ax.set_title("Median movie runtime by release decade")
ax.set_xlabel("Release decade")
ax.set_ylabel("Median runtime (minutes)")
save_chart("08_runtime_by_decade")

### Q8. How many seasons do TV shows have?

In [ ]:
tv = clean[clean["type"] == "TV Show"]
season_counts = tv["tv_seasons"].dropna().astype(int).value_counts().sort_index()
one_season_share = (tv["tv_seasons"] == 1).mean() * 100

ax = season_counts.plot(kind="bar", color=SERIES_2, rot=0)
ax.set_title(f"{one_season_share:.0f}% of TV shows have just one season")
ax.set_xlabel("Number of seasons")
ax.set_ylabel("Number of TV shows")
save_chart("09_tv_seasons")

### Q9. Which countries produce the most titles?

In [ ]:
top_countries = countries["country"].value_counts().head(10).sort_values()

ax = top_countries.plot(kind="barh", color=NETFLIX_RED)
ax.bar_label(ax.containers[0], padding=3)
ax.set_title(f"{top_countries.index[-1]} is credited on the most titles")
ax.set_xlabel("Number of titles (co-productions count for each country)")
ax.set_ylabel("")
save_chart("10_top_countries")

### Q10. Who are the titles made for?
`pd.crosstab` counts combinations of two columns. With `normalize="index"` each row becomes percentages that add up to 100, which lets us compare movies and TV shows fairly even though their totals differ.

In [ ]:
order = ["Kids", "Older Kids", "Teens", "Adults", "Unrated / Other"]
audience_share = (
    pd.crosstab(clean["type"], clean["audience"], normalize="index")
    .reindex(columns=order, fill_value=0)
    * 100
)

ax = audience_share.plot(
    kind="barh", stacked=True, figsize=(10, 3.5),
    color=["#f5b7b1", "#e6786f", "#E50914", "#8b0000", "#bdbdbd"],
)
ax.set_title("Audience mix of movies vs TV shows (% of titles)")
ax.set_xlabel("Share of titles (%)")
ax.set_ylabel("")
ax.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left")
save_chart("11_audience_mix")

## Step 5: Turn numbers into a story

Charts alone are not an analysis. A reader needs the *so what*. The cell below pulls the key numbers straight from your results, so your written findings always match your data. Copy these into your report and add your own interpretation.

In [ ]:
print("KEY FINDINGS")
print("-" * 60)
print(f"1. Catalog size: {len(clean):,} titles "
      f"({type_share.get('Movie', 0)}% movies, {type_share.get('TV Show', 0)}% TV shows).")
print(f"2. Release years: the busiest single year was {peak_year}; "
      f"the busiest decade was the {top_decade}s.")
print(f"3. Arrival on Netflix: the most titles were added in {peak_added}.")
print(f"4. Typical movie: about {median_runtime:.0f} minutes long.")
print(f"5. TV shows: {one_season_share:.0f}% have only one season.")
print(f"6. Most common genre label: {top_genres.index[-1]}.")
print(f"7. Most credited country: {top_countries.index[-1]}.")

## How to present this for a class, portfolio or interview

### The storytelling formula: *Question, Evidence, So what*
For each finding, write three short lines:
- **Question:** "Does Netflix have more movies or TV shows?"
- **Evidence:** the chart plus one number ("X% are movies").
- **So what:** what it suggests, with a caution ("This is a 2021 snapshot, so it may not hold today").

### Chart checklist
- [ ] The **title states the finding**, not just the topic
- [ ] Both axes are labelled, with units
- [ ] One message per chart; remove anything that does not support it
- [ ] Use at most 2 or 3 colors; keep the same color for the same thing across charts
- [ ] Sort bars so the biggest is at one end
- [ ] Add data labels where precise values matter

### A simple 8-slide structure
1. **Title and goal** - the dataset and your main question
2. **Data overview** - rows, columns, source, date of the snapshot
3. **Cleaning summary** - the problems you found and how you fixed them (use the before/after table)
4. **Catalog mix** - movies vs TV shows (Q1)
5. **Time trends** - releases by year/decade and when titles were added (Q2 to Q4)
6. **Content** - genres, countries, audience (Q5, Q6, Q9, Q10)
7. **Length** - movie runtimes and TV seasons (Q7, Q8)
8. **Conclusions and limitations** - key findings and next steps

### Limitations to state honestly (this earns trust)
- The data is a **snapshot**, not the live catalog.
- A catalog shows what Netflix *carries*, not what exists in the world or what people *watch*.
- Genre and country counts include **multi-label titles**, so one title can be counted several times.
- Missing director/cast/country values were labelled "Unknown", not guessed.
- Patterns are **descriptive**; this analysis cannot explain *why* they happen.

### Publishing it
- Put the notebook, `netflix_clean.csv` (or a link to the source) and the `charts/` folder on **GitHub**.
- Add a short **README**: the question, dataset source, 3 key findings with one chart each, and how to run it.

## Pandas cheat sheet: what you used

| Task | Code |
|---|---|
| Read a CSV | `pd.read_csv("file.csv")` |
| Peek at the data | `df.head()`, `df.info()`, `df.describe()` |
| Count missing values | `df.isna().sum()` |
| Find duplicates | `df.duplicated().sum()` |
| Count each value | `df["col"].value_counts()` |
| Remove stray spaces | `df["col"].str.strip()` |
| Fill missing values | `df["col"].fillna("Unknown")` |
| Text to dates | `pd.to_datetime(df["col"], errors="coerce")` |
| Get year from date | `df["date"].dt.year` |
| Extract numbers from text | `df["col"].str.extract(r"(\d+)")` |
| Recode categories | `df["col"].map({...})` |
| Filter rows | `df[df["col"] == "value"]` |
| Group and summarize | `df.groupby("col")["num"].median()` |
| Split a cell into rows | `.str.split(",")` then `.explode("col")` |
| Cross-tabulate | `pd.crosstab(df["a"], df["b"], normalize="index")` |

## Stretch challenges (when you are ready)

1. **Directors:** which directors have the most titles? (You will need to split and explode `director` like we did for genres, and ignore "Unknown".)
2. **Seasonality:** use `month_added` to find which months Netflix adds the most titles. Try a bar chart ordered January to December.
3. **Real clustering:** the decade grouping is manual. Try `sklearn.cluster.KMeans` on `release_year` and `movie_minutes` to let an algorithm find natural groups, then describe each cluster.
4. **Text analysis:** count the most common words in `description` per genre, or use TF-IDF to find what makes genres sound different.
5. **Compare platforms:** repeat the project with a Spotify dataset (track duration, release year, genre) and compare the patterns.
6. **Dashboard:** rebuild your best three charts as an interactive dashboard in Streamlit, Power BI or Tableau.